# BPOM Data Validation

This notebook validates the downloaded BPOM dataset for duplicates, NIE uniqueness, and API fidelity (ensuring the saved data exactly matches the API's format without corruption).


## 1. Setup & Imports


In [4]:
import pandas as pd
import glob
import os
import requests
from dotenv import load_dotenv

load_dotenv()

DATA_DIR = "Data BPOM"
BPOM_ACCESS_TOKEN = os.getenv("BPOM_ACCESS_TOKEN")

if not BPOM_ACCESS_TOKEN:
    print("WARNING: BPOM_ACCESS_TOKEN is not set. The API Fidelity Check will fail.")

print("Setup complete.")


Setup complete.


## 2. Data Loading & Aggregation


In [5]:
# Load all CSV files
csv_files = glob.glob(os.path.join(DATA_DIR, "bpom_products_*.csv"))
csv_files.sort()

print(f"Found {len(csv_files)} CSV files. Loading...")

dfs = []
for file in csv_files:
    # Use dtype=str for all columns to prevent Pandas from converting strings with leading zeros to ints/floats
    df_chunk = pd.read_csv(file, dtype=str)
    dfs.append(df_chunk)

if dfs:
    df = pd.concat(dfs, ignore_index=True)
    print(f"Successfully loaded a total of {len(df):,} records with {len(df.columns)} columns.")
    display(df.head())
else:
    print("No CSV files found in the directory.")
    df = pd.DataFrame()


Found 5 CSV files. Loading...
Successfully loaded a total of 661,894 records with 18 columns.


,id,no_aju,nama_produk,brand,nie,kode_qr,tanggal_terbit,masa_berlaku,nama_komoditi,bentuk_sediaan,kemasan,komposisi,pendaftar,negara_pendaftar,khasiat_produk,status_produk,nama_aplikasi,diterbitkan_oleh
0,1,ERBA300813202200011,Saus Bangkok Pedas Manis,FINNA,MD 243771002000149,(90)MD243771002000149,2023-07-18,2028-07-18,Pangan Olahan,Saus Bumbu,"Plastik/ Aluminium Foil (120 ml, 135ml, 250 m...",NaN,SEKAR LAUT,Indonesia,NaN,Berlaku,e-Registration Pangan Olahan RBA (e-Reg RBA),Direktorat Registrasi Pangan Olahan
1,2,ERBA300652202300001,Abon Ikan Tuna dan Tongkol,Wishfood Abon,MD 042358000300007,(90)MD042358000300007,2023-03-01,2028-03-01,Pangan Olahan,Pangan kategori 9 risiko rendah lainnya,aluminium foil (100 gram),NaN,BEEHIVE CORP INDONESIA,Indonesia,NaN,Berlaku,e-Registration Pangan Olahan RBA (e-Reg RBA),Direktorat Registrasi Pangan Olahan
2,3,ERBA300280202500424,Mi Instan Cup Ramen Kuah Miso Rasa Ayam,POP MIE,MD 240935012400027,(90)MD240935012400027,2025-07-17,2030-06-16,Pangan Olahan,Pasta dan mi kering serta produk sejenis,Cup Kertas/Plastik PE (77 g),NaN,INDOFOOD CBP SUKSES MAKMUR,Indonesia,NaN,Berlaku,e-Registration Pangan Olahan RBA (e-Reg RBA),Direktorat Registrasi Pangan Olahan
3,4,ERBA300186202400221,"Anggur Merah- (Mengandung Alkohol ±12,5% v/v )...",Raisins Gaulois,ML 210982049000170,(90)ML210982049000170,2024-08-26,2029-08-26,Pangan Olahan,Minuman beralkohol,Botol Kaca (750ml),NaN,PANTJA ARTHA NIAGA,Indonesia,NaN,Berlaku,e-Registration Pangan Olahan RBA (e-Reg RBA),Direktorat Registrasi Pangan Olahan
4,5,ERBA3113716202600024,"Es Krim Rasa Cokelat, Stroberi dan Vanila",WALLS,MD 242811006301471,(90)MD242811006301471,2026-03-31,2031-03-31,Pangan Olahan,Es krim,Plastik Laminat (66ml),NaN,THE MAGNUM ICE CREAM INDONESIA,Indonesia,NaN,Berlaku,e-Registration Pangan Olahan RBA (e-Reg RBA),Direktorat Registrasi Pangan Olahan


## 3. Duplicate Analysis


In [6]:
if not df.empty:
    print("--- 1. Full Row Duplicates Check ---")
    exact_duplicates = df.duplicated().sum()
    print(f"Exact row duplicates found: {exact_duplicates:,}")
    
    print("\n--- 2. ID Duplicates Check ---")
    # Identify the primary key column, usually 'id' or 'id_produk'
    id_col = 'id' if 'id' in df.columns else None
    if not id_col:
        print("Warning: No 'id' column found!")
    else:
        id_duplicates = df.duplicated(subset=[id_col]).sum()
        print(f"Duplicate IDs found: {id_duplicates:,}")
        
        if id_duplicates > 0:
            print("\nSample of duplicated IDs:")
            display(df[df.duplicated(subset=[id_col], keep=False)].sort_values(by=id_col).head(10))


--- 1. Full Row Duplicates Check ---
Exact row duplicates found: 0

--- 2. ID Duplicates Check ---
Duplicate IDs found: 0


## 4. NIE Uniqueness Check


In [7]:
if not df.empty and 'nie' in df.columns:
    unique_nies = df['nie'].nunique()
    total_records = len(df)
    
    print(f"Total Unique NIEs: {unique_nies:,} out of {total_records:,} total records.")
    
    # Calculate how many NIEs are duplicated
    nie_counts = df['nie'].value_counts()
    duplicated_nies = nie_counts[nie_counts > 1]
    
    print(f"Number of NIEs that are assigned to multiple records: {len(duplicated_nies):,}")
    
    if len(duplicated_nies) > 0:
        print("\nNote: It is normal for an NIE to be assigned to multiple records if a product has multiple packaging variants (kemasan).")
        print("Here are the top 3 most frequently duplicated NIEs to verify this:")
        
        for idx, (nie, count) in enumerate(duplicated_nies.head(3).items()):
            print(f"\n--- Top {idx+1}: NIE {nie} appears {count} times ---")
            cols_to_show = ['id', 'nama_produk', 'brand', 'pendaftar', 'kemasan']
            # Filter to available columns
            cols_to_show = [c for c in cols_to_show if c in df.columns]
            
            sample_df = df[df['nie'] == nie][cols_to_show].head(5)
            display(sample_df)


Total Unique NIEs: 516,377 out of 661,894 total records.
Number of NIEs that are assigned to multiple records: 130,058

Note: It is normal for an NIE to be assigned to multiple records if a product has multiple packaging variants (kemasan).
Here are the top 3 most frequently duplicated NIEs to verify this:

--- Top 1: NIE MD 242888003000044 appears 52 times ---


,id,nama_produk,brand,pendaftar,kemasan
558,559,Makanan Ringan Mi Rasa Ayam Geprek,INDO KRIP KRIP,INDOFOOD CBP SUKSES MAKMUR,Plastik Metalized (18 g)
2510,2511,Makanan Ringan Mi Rasa Ayam Geprek,ANAK MAS,INDOFOOD CBP SUKSES MAKMUR,Plastik Metalized (18 g)
2787,2788,Makanan Ringan Mi Rasa Ayam Geprek,INDO KRIP KRIP,INDOFOOD CBP SUKSES MAKMUR,Plastik Metalized (18 g)
8720,11912,Makanan Ringan Mi Rasa Ayam Geprek,INDO KRIP KRIP,INDOFOOD CBP SUKSES MAKMUR,Plastik Metalized (18 g)
9614,12806,Makanan Ringan Mi Rasa Ayam Geprek,INDO KRIP KRIP,INDOFOOD CBP SUKSES MAKMUR,Plastik Metalized (18 g)



--- Top 2: NIE MD 220935002300027 appears 35 times ---


,id,nama_produk,brand,pendaftar,kemasan
4843,4844,Mi Telur (Folded),CAP 3 AYAM,INDOFOOD CBP SUKSES MAKMUR,Plastik PP (550 g)
14153,17345,Mi Telur (Folded),CAP 3 AYAM,INDOFOOD CBP SUKSES MAKMUR,Plastik PP (550 g)
19891,23083,Mi Telur (Folded),CAP 3 AYAM,INDOFOOD CBP SUKSES MAKMUR,Plastik PP (550 g)
20851,27234,Mi Telur (Folded),CAP 3 AYAM,INDOFOOD CBP SUKSES MAKMUR,Plastik PP (550 g)
21416,27799,Mi Telur (Folded),CAP 3 AYAM,INDOFOOD CBP SUKSES MAKMUR,Plastik PP (550 g)



--- Top 3: NIE MD 231509401002 appears 35 times ---


,id,nama_produk,brand,pendaftar,kemasan
164291,260022,Mi Instan Goreng Rasa Nasi Padang,SARIMI (Puass),PT. Indofood CBP Sukses Makmur Tbk,Plastik (100 g)
185811,297497,Mi Instan Goreng Rasa Nasi Padang,SARIMI (Puass),PT. Indofood CBP Sukses Makmur Tbk,Plastik (100 g)
195718,313786,Mi Instan Goreng Rasa Nasi Padang,SARIMI (Puass),PT. Indofood CBP Sukses Makmur Tbk,Plastik (100 g)
219458,356672,Mi Instan Goreng Rasa Ayam Kremess,SARIMI,PT. Indofood CBP Sukses Makmur Tbk,Plastik PP (70 g)
244730,397899,Mi Instan Goreng Rasa Nasi Padang,SARIMI (Puass),PT. Indofood CBP Sukses Makmur Tbk,Plastik (100 g)


## 5. API Fidelity Check


In [8]:
def run_fidelity_check():
    if df.empty:
        print("DataFrame is empty. Cannot run fidelity check.")
        return False
        
    if not BPOM_ACCESS_TOKEN:
        print("Cannot run fidelity check without BPOM_ACCESS_TOKEN.")
        return False
        
    print("Fetching sample from API for Fidelity Check...")
    
    # Fetch offset 0, limit 100
    API_BASE_URL = "https://satudata.pom.go.id/api/items/app_webreg_masterproduk"
    params = {
        "fields": "*",
        "limit": 100,
        "offset": 0,
        "access_token": BPOM_ACCESS_TOKEN
    }
    
    try:
        response = requests.get(API_BASE_URL, params=params, timeout=30)
        response.raise_for_status()
        api_data = response.json().get("data", [])
    except Exception as e:
        print(f"Failed to fetch data from API: {e}")
        return False
        
    if not api_data:
        print("API returned no data.")
        return False
        
    print(f"Successfully fetched {len(api_data)} sample records from API.")
    
    # Check column by column
    all_passed = True
    mismatches = []
    
    for api_record in api_data:
        record_id = str(api_record.get('no_aju'))
        
        # Find this record in the dataframe
        matched_rows = df[df['no_aju'] == record_id]
        if matched_rows.empty:
            print(f"❌ Record ID {record_id} not found in CSV!")
            all_passed = False
            continue
            
        csv_record = matched_rows.iloc[0].to_dict()
        
        # Compare fields
        for key, api_val in api_record.items():
            if key == 'id':
                continue
            if key not in csv_record:
                mismatches.append(f"ID {record_id}: Missing column '{key}' in CSV.")
                all_passed = False
                continue
                
            csv_val = csv_record[key]
            
            # Handle nulls
            if api_val is None:
                if pd.notna(csv_val):
                    if str(csv_val).strip() != "":
                        mismatches.append(f"ID {record_id} [{key}]: API had null, CSV has '{csv_val}'")
                        all_passed = False
                continue
            
            api_val_str = str(api_val)
            
            if api_val_str != csv_val:
                mismatches.append(f"ID {record_id} [{key}]: API='{api_val_str}' vs CSV='{csv_val}'")
                all_passed = False
                
    if all_passed:
        print("\n✅ FIDELITY CHECK PASSED: Data in CSV perfectly matches the API.")
        print("No characters, leading zeros, or formatting (like (90)MD...) were lost during ingestion.")
        return True
    else:
        print("\n❌ FIDELITY CHECK FAILED:")
        for m in mismatches[:10]: # Print up to 10 mismatches
            print(m)
        if len(mismatches) > 10:
            print(f"...and {len(mismatches) - 10} more mismatches.")
        return False

# Run the check
fidelity_result = run_fidelity_check()


Fetching sample from API for Fidelity Check...
Successfully fetched 100 sample records from API.

❌ FIDELITY CHECK FAILED:
ID 1 [no_aju]: API='ERBA300328202300365' vs CSV='ERBA300813202200011'
ID 1 [nama_produk]: API='Minuman Serbuk Instan Rasa Cocopandan' vs CSV='Saus Bangkok Pedas Manis'
ID 1 [brand]: API='Nutrimas' vs CSV='FINNA'
ID 1 [nie]: API='MD 243182003600019' vs CSV='MD 243771002000149'
ID 1 [kode_qr]: API='(90)MD243182003600019' vs CSV='(90)MD243771002000149'
ID 1 [tanggal_terbit]: API='2023-12-19' vs CSV='2023-07-18'
ID 1 [masa_berlaku]: API='2028-09-05' vs CSV='2028-07-18'
ID 1 [bentuk_sediaan]: API='Minuman serbuk' vs CSV='Saus Bumbu'
ID 1 [kemasan]: API='Plastik Laminat (7g)' vs CSV='Plastik/ Aluminium  Foil (120 ml, 135ml, 250 ml, 275 ml)'
ID 1 [komposisi]: API='' vs CSV='nan'
...and 1186 more mismatches.
